# PROYECTO INTEGRADO - ULEAM 2026
## FASE 2: PIPELINE DE INTEGRACIÓN (ETL) Y CALIDAD DE DATOS
### Autores
- CARDENAS AVILA EMILIO SLEIMEN
- INTRIAGO BENÍTEZ MICHAEL AGUSTÍN
- MEJIA ORDOÑEZ ANTHONY AXEL


# 0. IMPORTAR LIBRERIAS

In [ ]:
import pandas as pd
import numpy as np

print("--- INICIANDO EXTRACCIÓN Y SEPARACIÓN MULTIDIMENSIONAL ---")

--- INICIANDO EXTRACCIÓN Y SEPARACIÓN MULTIDIMENSIONAL ---


# 1. CARGA DE DATOS ORIGEN


In [ ]:
archivo_origen = '/content/mdi_personasdesaparecidas_pm_2017_2025.xlsx'
df = pd.read_excel(archivo_origen, sheet_name='1. Pdesaparecidas')
print(f"[OK] Dataset cargado correctamente. Registros iniciales: {df.shape[0]}")

# Crear copia de trabajo
df_clean = df.copy()

[OK] Dataset cargado correctamente. Registros iniciales: 75699


# PASO 2: TRATAMIENTO DE CALIDAD DE DATOS (LIMPIEZA GENERAL)

In [ ]:
# A. Corrección de Coordenadas Geográficas (De texto con comas a decimales con puntos)
print("-> Normalizando coordenadas geográficas...")
df_clean['latitud_desaparicion'] = pd.to_numeric(df_clean['latitud_desaparicion'].astype(str).str.replace(',', '.'), errors='coerce')
df_clean['longitud_desaparicion'] = pd.to_numeric(df_clean['longitud_desaparicion'].astype(str).str.replace(',', '.'), errors='coerce')
df_clean['latitud_localizacion'] = pd.to_numeric(df_clean['latitud_localizacion'].astype(str).str.replace(',', '.'), errors='coerce')
df_clean['longitud_localizacion'] = pd.to_numeric(df_clean['longitud_localizacion'].astype(str).str.replace(',', '.'), errors='coerce')

# B. Corrección de la variable Edad (Convertir a formato entero e imputar nulos)
print("-> Transformando variable Edad...")
df_clean['edad'] = pd.to_numeric(df_clean['edad'].astype(str).str.replace(',', '.'), errors='coerce')
mediana_edad = df_clean['edad'].median()
df_clean['edad'] = df_clean['edad'].fillna(mediana_edad).astype(int)

# C. Solución Métrico Temporal (Conversión de segundos a días exactos)
print("-> Recalculando métrica temporal de 'dias_solucion'...")
df_clean['dias_solucion_limpio'] = pd.to_numeric(df_clean['dias_solucion'].astype(str).str.replace(',', '.'), errors='coerce')
df_clean['dias_solucion_dias'] = df_clean['dias_solucion_limpio'] / 86400
# Limpieza de anomalías (valores negativos erróneos en la base origen)
df_clean.loc[df_clean['dias_solucion_dias'] < 0, 'dias_solucion_dias'] = np.nan

# D. Estandarización de Textos (Tratamiento de nulos conceptuales)
print("-> Estandarizando cadenas de texto 'NO_APLICA'...")
columnas_localizacion = ['fecha_localizacion', 'latitud_localizacion', 'longitud_localizacion', 'provincia_localizacion']
for col in columnas_localizacion:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].replace('NO_APLICA', np.nan)

-> Normalizando coordenadas geográficas...
-> Transformando variable Edad...
-> Recalculando métrica temporal de 'dias_solucion'...
-> Estandarizando cadenas de texto 'NO_APLICA'...


# PASO 3: EXTRACCIÓN Y GENERACIÓN DE LAS TABLAS DE DIMENSIONES (ESQUEMA ESTRELLA)

In [ ]:
print("\n--- CONSTRUYENDO ENTIDADES DEL MODELO MULTIDIMENSIONAL ---")

# A. Dimensión Geografía: Filtramos las columnas geográficas y removemos duplicados
print("-> Generando Dim_Geografia...")
cols_geo = ['circuito', 'subcircuito', 'distrito', 'zona', 'canton', 'codigo_canton', 'provincia', 'codigo_provincia']
dim_geografia = df_clean[cols_geo].drop_duplicates().dropna(subset=['circuito'])

# B. Dimensión Víctima: Filtramos variables demográficas y removemos duplicados
print("-> Generando Dim_Victima...")
cols_vic = ['sexo', 'rango_edad', 'etnia', 'nacionalidad']
dim_victima = df_clean[cols_vic].drop_duplicates().dropna(subset=['sexo'])

# C. Dimensión Caso: Aislamos variables contextuales del estado de investigación y removemos duplicados
print("-> Generando Dim_Caso...")
cols_caso = ['estado_desaparecido', 'situacion_actual', 'motivo_desaparicion', 'motivacion_desaparicion_observada']
dim_caso = df_clean[cols_caso].drop_duplicates().dropna(subset=['estado_desaparecido'])


--- CONSTRUYENDO ENTIDADES DEL MODELO MULTIDIMENSIONAL ---
-> Generando Dim_Geografia...
-> Generando Dim_Victima...
-> Generando Dim_Caso...


# PASO 4: CONSTRUCCIÓN DE LA TABLA DE HECHOS CENTRAL (FACT TABLE)

In [ ]:
print("-> Generando Fact_Desapariciones...")
# La tabla central mantiene solo las llaves, las fechas, las métricas y las coordenadas exactas por fila
cols_fact = [
    'circuito', 'sexo', 'estado_desaparecido',  # Llaves relacionales
    'fecha_desaparicion', 'fecha_denuncia', 'fecha_conocimiento', 'fecha_localizacion',  # Atributos temporales
    'latitud_desaparicion', 'longitud_desaparicion', 'latitud_localizacion', 'longitud_localizacion',  # Ubicaciones Geo
    'edad', 'dias_solucion_dias', 'provincia_localizacion'  # Métricas operativas
]
fact_desapariciones = df_clean[cols_fact]

-> Generando Fact_Desapariciones...


# PASO 5: EXPORTACIÓN DE LOS PRODUCTOS DE DATOS (4 ARCHIVOS CSV INDEPENDIENTES)

In [ ]:
print("\n--- EXPORTANDO ARCHIVOS FINALES PARA POWER BI ---")
dim_geografia.to_csv('Dim_Geografia.csv', index=False, encoding='utf-8')
dim_victima.to_csv('Dim_Victima.csv', index=False, encoding='utf-8')
dim_caso.to_csv('Dim_Caso.csv', index=False, encoding='utf-8')
fact_desapariciones.to_csv('Fact_Desapariciones.csv', index=False, encoding='utf-8')

print("\n--- PIPELINE FINALIZADO CON ÉXITO ---")
print("[INFO] Se han generado y guardado 4 archivos CSV en tu panel izquierdo.")
print(f"- Dim_Geografia.csv    | Filas únicas: {dim_geografia.shape[0]}")
print(f"- Dim_Victima.csv      | Filas únicas: {dim_victima.shape[0]}")
print(f"- Dim_Caso.csv         | Filas únicas: {dim_caso.shape[0]}")
print(f"- Fact_Desapariciones.csv | Registros enlazados: {fact_desapariciones.shape[0]}")


--- EXPORTANDO ARCHIVOS FINALES PARA POWER BI ---

--- PIPELINE FINALIZADO CON ÉXITO ---
[INFO] Se han generado y guardado 4 archivos CSV en tu panel izquierdo.
- Dim_Geografia.csv    | Filas únicas: 1809
- Dim_Victima.csv      | Filas únicas: 346
- Dim_Caso.csv         | Filas únicas: 66
- Fact_Desapariciones.csv | Registros enlazados: 75699
